# v2-01 · Idle-delay quasi-inverse on hardware (single delays)

**What changed vs. the draft-7 notebook (`Delay_Capturing_single_delay_QI.ipynb`)**

1. The closed-form $Q$ matrix was wrong (it is not the quadratic form of $\Delta D^2$; it predicted a reduction larger than the raw MSTD). Replaced by the verified `quasi_inverse()` routine (self-tested against brute force every time the notebook runs).
2. Several delays are measured in one run (`DELAYS_DT`), all circuits of a kind go in one job to limit drift.
3. Validation uses all six Pauli eigenstates, raw **and** corrected, so we can report the *measured* corrected MSTD $D^2$ (Bloch-ball average), not just $D$ for $|+\rangle$.
4. Shot-noise standard errors on every number; backend calibration (T1, T2, readout error, dt) and job IDs are recorded.
5. No token in the notebook; `USE_SIMULATOR=True` runs the whole flow on a noise model of ibm_fez first.

**What to send back:** the file `results_single_delay_<backend>.json` that the last cell writes (it contains everything, including raw counts), plus the printed summary table.


In [ ]:
EXTRA_PKGS = []
# Colab / fresh environment: install once (comment out if already installed)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "qiskit>=1.4", "qiskit-aer", "qiskit-ibm-runtime>=0.40", "matplotlib", "pandas"] + EXTRA_PKGS,
               check=False)


In [ ]:
from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService

# Retrieve the token securely from Colab Secrets
ibm_token = userdata.get("IBM_QUANTUM_TOKEN")

# Save the account to the environment
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token=ibm_token,
    overwrite=True,
)
print("Account saved for this environment.")

## Shared, verified MSTD / quasi-inverse routines (identical in all v2 notebooks)

In [ ]:
# =============================================================================
#  MSTD quasi-inverse: shared, verified routines  (v2, 2026-11-05)
#  This block is pasted verbatim into every v2 notebook. Do not edit one copy
#  without editing the others.
#
#  Conventions (identical to the paper):
#    rho(r) = (I + r.sigma)/2,   channel  r -> M r + c
#    D^2(M,c) = [Tr(M M^T) - 2 Tr M + 3]/20 + |c|^2/4           (Bloch-ball MSTD)
#    V(x) = x0 I + i (x1 X + x2 Y + x3 Z),   |x| = 1             (unitary recovery)
#    V rho(r) V^dag = rho(R_V r),   V o E  ->  (R_V M, R_V c)
#
#  Key identity used for the quasi-inverse:
#    D^2(R M, R c) = D^2(M,c) - (1/10)[Tr(R M) - Tr M]           (R orthogonal)
#  so maximising the MSTD reduction  Delta D^2 = (1/10)[Tr(R_V M) - Tr M]
#  is the same as maximising Tr(R_V M), which is an exact quadratic form in x:
#    Delta D^2(x) = (1/5) x^T Qhat x,
#    Qhat_00 = 0,  Qhat_vv = (M + M^T)/2 - Tr(M) I_3,
#    Qhat_01 = -(M_23 - M_32)/2, Qhat_02 = -(M_31 - M_13)/2, Qhat_03 = -(M_12 - M_21)/2
#  (indices 1,2,3 = x,y,z).  The optimum is the top eigenvector of Qhat and
#  Delta D^2_max = lambda_max(Qhat)/5.  Because the identity (x = e_0) gives
#  x^T Qhat x = 0, lambda_max >= 0 always, and lambda_max = 0 means "do nothing".
#
#  NOTE: the expression in draft 7 of the paper (prefactor 2/5, diagonal block
#  (M+M^T)/2 - Tr(M)/2 I, off-diagonals -1/4 (...)) is NOT the quadratic form of
#  Delta D^2; self_test() below checks the one used here against brute force.
# =============================================================================
import json, math, time, datetime
import numpy as np

I2 = np.eye(2, dtype=complex)
PX = np.array([[0, 1], [1, 0]], dtype=complex)
PY = np.array([[0, -1j], [1j, 0]], dtype=complex)
PZ = np.array([[1, 0], [0, -1]], dtype=complex)
PAULIS = [PX, PY, PZ]


def mstd(M, c):
    """Bloch-ball mean-square trace distance of the affine channel (M, c)."""
    M = np.asarray(M, float); c = np.asarray(c, float).reshape(3)
    return float((np.trace(M @ M.T) - 2.0 * np.trace(M) + 3.0) / 20.0 + c @ c / 4.0)


def unitary_from_x(x):
    """V = x0 I + i(x1 X + x2 Y + x3 Z), normalised."""
    x = np.asarray(x, float); x = x / np.linalg.norm(x)
    return x[0] * I2 + 1j * (x[1] * PX + x[2] * PY + x[3] * PZ)


def rotation_from_unitary(V):
    """SO(3) matrix R_V with V rho(r) V^dag = rho(R_V r)."""
    return np.array([[0.5 * np.trace(si @ V @ sj @ V.conj().T).real for sj in PAULIS] for si in PAULIS])


def build_Qhat(M):
    """Exact 4x4 symmetric matrix with Delta D^2(x) = (1/5) x^T Qhat x."""
    M = np.asarray(M, float)
    Q = np.zeros((4, 4))
    Q[1:, 1:] = 0.5 * (M + M.T) - np.trace(M) * np.eye(3)
    Q[0, 1] = Q[1, 0] = -(M[1, 2] - M[2, 1]) / 2.0
    Q[0, 2] = Q[2, 0] = -(M[2, 0] - M[0, 2]) / 2.0
    Q[0, 3] = Q[3, 0] = -(M[0, 1] - M[1, 0]) / 2.0
    return Q


def quasi_inverse(M, c, min_gain=1e-4):
    """Optimal unitary quasi-inverse of the affine channel (M, c).

    Returns dict with x, V, R, Q, lambda_max, D2_raw, D2_corrected, delta_D2 (= delta_D2_predicted).
    If the best achievable reduction is below `min_gain` the identity is returned (nothing to correct).
    """
    M = np.asarray(M, float); c = np.asarray(c, float).reshape(3)
    Q = build_Qhat(M)
    evals, evecs = np.linalg.eigh(Q)
    lam = float(evals[-1]); x = np.real(evecs[:, -1]); x /= np.linalg.norm(x)
    if lam / 5.0 <= min_gain:           # channel is already best left alone
        lam = 0.0; x = np.array([1.0, 0.0, 0.0, 0.0])
    if x[0] < 0: x = -x                 # fix the global-phase sign convention
    V = unitary_from_x(x); R = rotation_from_unitary(V)
    D2_raw = mstd(M, c); D2_qi = mstd(R @ M, R @ c)
    out = dict(x=x, V=V, R=R, Q=Q, lambda_max=lam, D2_raw=D2_raw,
               D2_corrected=D2_qi, delta_D2=D2_raw - D2_qi, delta_D2_predicted=lam / 5.0)
    # internal consistency: the closed form must reproduce the direct evaluation
    assert abs(out["delta_D2"] - out["delta_D2_predicted"]) < 1e-9, "Qhat inconsistent with direct MSTD"
    assert out["delta_D2"] <= D2_raw + 1e-12, "reduction larger than the raw MSTD (impossible)"
    assert np.linalg.norm(V.conj().T @ V - I2) < 1e-12
    return out


def affine_from_tomography(r0, r1, rp, ry):
    """(M, c) from the Bloch vectors of the outputs for inputs |0>,|1>,|+>,|+y>."""
    r0, r1, rp, ry = (np.asarray(v, float) for v in (r0, r1, rp, ry))
    c = 0.5 * (r0 + r1)
    M = np.column_stack([rp - c, ry - c, 0.5 * (r0 - r1)])
    return M, c


def affine_from_pairs(r_in, r_out):
    """Least-squares (M, c) from any set of (input, output) Bloch vectors (>= 4 inputs)."""
    r_in = np.asarray(r_in, float); r_out = np.asarray(r_out, float)
    A = np.hstack([r_in, np.ones((len(r_in), 1))])           # [r | 1]
    sol, *_ = np.linalg.lstsq(A, r_out, rcond=None)          # sol^T = [M | c]
    return sol[:3].T, sol[3]


def expectation_and_se(counts, key0="0", key1="1"):
    """<P> = (n0 - n1)/N and its binomial standard error."""
    n0 = counts.get(key0, 0); n1 = counts.get(key1, 0); N = n0 + n1
    if N == 0: return float("nan"), float("nan")
    p = n0 / N; e = 2 * p - 1
    se = 2.0 * math.sqrt(max(p * (1 - p), 0.25 / N) / N)     # floor avoids se = 0 at p in {0,1}
    return e, se


def trace_distance(r1, r2):
    return 0.5 * float(np.linalg.norm(np.asarray(r1, float) - np.asarray(r2, float)))


def trace_distance_se(r1, r2, se2):
    """Propagated 1-sigma error of D(r1, r2) when r2 has component errors se2."""
    d = np.asarray(r1, float) - np.asarray(r2, float); n = np.linalg.norm(d)
    if n < 1e-12: return 0.5 * float(np.linalg.norm(se2))
    return 0.5 * float(math.sqrt(np.sum((d / n) ** 2 * np.asarray(se2, float) ** 2)))


def sampled_mstd(M, c, n=20000, seed=0):
    """Monte-Carlo D^2 over the uniform Bloch ball (cross-check of the closed form)."""
    rng = np.random.default_rng(seed)
    v = rng.normal(size=(n, 3)); v /= np.linalg.norm(v, axis=1)[:, None]
    r = v * rng.uniform(0, 1, n)[:, None] ** (1 / 3)
    d = r - (r @ np.asarray(M, float).T + np.asarray(c, float))
    return float(np.mean(np.sum(d * d, axis=1)) / 4.0)


def backend_metadata(backend, qubits):
    """Backend name, dt, and T1/T2/readout/gate errors for the physical qubits used."""
    meta = {"backend": getattr(backend, "name", str(backend)), "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "qubits": list(qubits)}
    try:
        meta["dt_seconds"] = float(backend.dt) if backend.dt is not None else None
    except Exception:
        meta["dt_seconds"] = None
    try:
        props = backend.properties()
        meta["calibration"] = {}
        for q in qubits:
            meta["calibration"][int(q)] = {
                "T1_us": props.t1(q) * 1e6, "T2_us": props.t2(q) * 1e6,
                "readout_error": props.readout_error(q),
                "sx_error": props.gate_error("sx", [q]) if "sx" in {g.gate for g in props.gates} else None}
    except Exception as exc:
        meta["calibration_error"] = str(exc)
    return meta


class NpEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.ndarray):
            return o.tolist() if not np.iscomplexobj(o) else {"re": o.real.tolist(), "im": o.imag.tolist()}
        if isinstance(o, (np.floating, np.integer)): return o.item()
        if isinstance(o, complex): return {"re": o.real, "im": o.imag}
        return super().default(o)


def save_results(obj, filename):
    with open(filename, "w") as fh:
        json.dump(obj, fh, cls=NpEncoder, indent=1)
    print("saved", filename)
    try:                                   # Colab: trigger a download so the file can be sent back
        from google.colab import files
        files.download(filename)
    except Exception:
        pass


def self_test(n_random=200, seed=1):
    """Check Qhat against brute-force optimisation on random channels and on known cases."""
    rng = np.random.default_rng(seed)
    # exact quadratic form on random affine maps
    for _ in range(n_random):
        M = rng.normal(size=(3, 3)) * 0.5; c = rng.normal(size=3) * 0.1
        Q = build_Qhat(M)
        for _ in range(3):
            x = rng.normal(size=4); x /= np.linalg.norm(x)
            R = rotation_from_unitary(unitary_from_x(x))
            dd = mstd(M, c) - mstd(R @ M, R @ c)
            assert abs(dd - x @ Q @ x / 5.0) < 1e-10, "Qhat is not the quadratic form of Delta D^2"
        qi = quasi_inverse(M, c)
        best = min(mstd(rotation_from_unitary(unitary_from_x(v)) @ M, rotation_from_unitary(unitary_from_x(v)) @ c)
                   for v in rng.normal(size=(3000, 4)))
        assert qi["D2_corrected"] <= best + 1e-9, "eigenvector is not the optimum"
    # bit flip p=0.55: M=diag(1,-0.1,-0.1) -> V = X, Delta D^2 = 0.04
    qi = quasi_inverse(np.diag([1, -0.1, -0.1]), np.zeros(3))
    assert abs(qi["delta_D2"] - 0.04) < 1e-12 and abs(abs(qi["x"][1]) - 1) < 1e-9
    # near-identity channel -> V = identity
    qi = quasi_inverse(0.97 * np.eye(3), np.array([0, 0, 0.03]))
    assert qi["lambda_max"] == 0.0 and qi["x"][0] == 1.0
    # sampled MSTD agrees with closed form
    M = rng.normal(size=(3, 3)) * 0.4; c = rng.normal(size=3) * 0.1
    assert abs(sampled_mstd(M, c, 200000) - mstd(M, c)) < 2e-3
    print("mstd_qi_common self-test passed")



self_test()


## Backend

In [ ]:
# ---------------------------------------------------------------------------
# Backend selection.  NEVER paste a token into this notebook (two earlier
# notebooks had tokens committed in plain text -- those tokens must be revoked).
# Run ONCE in a separate scratch cell, then delete that cell:
#   from qiskit_ibm_runtime import QiskitRuntimeService
#   QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="...", overwrite=True)
# Alternatively set the environment variable IBM_QUANTUM_TOKEN (Colab: use userdata secrets).
# ---------------------------------------------------------------------------
import os
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.transpiler import generate_preset_pass_manager

USE_SIMULATOR = False            # <-- set False for the real QPU run
BACKEND_NAME  = "ibm_fez"       # the paper's primary device; keep all experiments on one device
SHOTS         = 4096

if USE_SIMULATOR:
    from qiskit_aer import AerSimulator
    from qiskit_ibm_runtime.fake_provider import FakeFez
    from qiskit_aer.primitives import SamplerV2 as Sampler
    fake = FakeFez()
    backend = AerSimulator.from_backend(fake)        # noise model + coupling map of ibm_fez
    backend.name = "aer(" + fake.name + ")"
    cal_backend = fake                               # properties() for metadata
    sampler = Sampler.from_backend(fake)
else:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    token = os.environ.get("IBM_QUANTUM_TOKEN")
    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token) if token else QiskitRuntimeService()
    backend = service.backend(BACKEND_NAME)
    cal_backend = backend
    sampler = Sampler(mode=backend)

print("Backend:", backend.name, "| simulator:", USE_SIMULATOR, "| shots:", SHOTS)

JOB_LOG = []   # every submitted job: id, purpose, number of circuits

def run_batch(circuits, purpose, physical_qubits=None, optimization_level=1):
    """Transpile + run one list of circuits; returns a list of counts dicts (same order)."""
    target = cal_backend if USE_SIMULATOR else backend
    pm = generate_preset_pass_manager(backend=target, optimization_level=optimization_level,
                                      initial_layout=physical_qubits)
    isa = pm.run(circuits)
    # sanity: delays must survive transpilation
    for q0, q1 in zip(circuits, isa):
        n_delay0 = sum(1 for inst in q0.data if inst.operation.name == "delay")
        n_delay1 = sum(1 for inst in q1.data if inst.operation.name == "delay")
        assert n_delay1 >= n_delay0, "transpiler removed a delay -- lower optimization_level"
    job = sampler.run(isa, shots=SHOTS)
    jid = job.job_id() if hasattr(job, "job_id") else "simulator"
    JOB_LOG.append({"job_id": jid, "purpose": purpose, "n_circuits": len(isa),
                    "depths": [c.depth() for c in isa], "two_qubit_gates": [c.num_nonlocal_gates() for c in isa],
                    "physical_qubits": [sorted({q._index if hasattr(q, "_index") else backend_index(c, q)
                                                for inst in c.data for q in inst.qubits if inst.operation.name == "measure"})
                                        for c in isa]})
    print(f"[{purpose}] job {jid}: {len(isa)} circuits submitted")
    res = job.result()
    out = []
    for pub in res:
        data = pub.data
        reg = [k for k in data.keys()][0]
        out.append(getattr(data, reg).get_counts())
    return out

def backend_index(circ, q):
    return circ.find_bit(q).index


## Tomography helpers

In [ ]:
# ---------------------------------------------------------------------------
# State preparation / measurement helpers (single logical qubit, register "meas")
# ---------------------------------------------------------------------------
PAULI_STATES = {            # name: (prep gates, Bloch vector)
    "0":  ([],            np.array([0, 0, 1.0])),
    "1":  (["x"],         np.array([0, 0, -1.0])),
    "+":  (["h"],         np.array([1.0, 0, 0])),
    "-":  (["x", "h"],    np.array([-1.0, 0, 0])),
    "+y": (["h", "s"],    np.array([0, 1.0, 0])),
    "-y": (["h", "sdg"],  np.array([0, -1.0, 0])),
}
CHAR_INPUTS = ["0", "1", "+", "+y"]      # minimal set that determines (M, c)
VAL_INPUTS  = list(PAULI_STATES)         # all six Pauli eigenstates for validation

def prepare(qc, q, state):
    for g in PAULI_STATES[state][0]:
        getattr(qc, g)(q)

def rotate_to_z(qc, q, basis):
    if basis == "X":   qc.h(q)
    elif basis == "Y": qc.sdg(q); qc.h(q)       # (H S^dag)^dag Z (H S^dag) = Y
    elif basis != "Z": raise ValueError(basis)

def bloch_from_counts(counts_xyz):
    """[counts_X, counts_Y, counts_Z] -> (r, se)"""
    vals = [expectation_and_se(c) for c in counts_xyz]
    return np.array([v[0] for v in vals]), np.array([v[1] for v in vals])


## Parameters

In [ ]:
QUBIT     = 0                                  # physical qubit (keep the same for all delay runs)
DELAYS_DT = [1000, 10000, 50000]               # idle times in units of dt (dt is read from the backend)
dt_s = getattr(cal_backend, "dt", None)
print("dt =", dt_s, "s  ->  delays in us:", [d * dt_s * 1e6 for d in DELAYS_DT] if dt_s else "unknown")


## Job 1 - characterise the idle channel

In [ ]:
# ---- Job 1: characterisation + raw validation (six inputs x three bases per delay) ----
def delay_circuit(state, basis, delay_dt, V=None):
    qr = QuantumRegister(1, "q"); cr = ClassicalRegister(1, "meas")
    qc = QuantumCircuit(qr, cr)
    prepare(qc, qr[0], state)
    qc.barrier()
    qc.delay(delay_dt, qr[0], unit="dt")
    qc.barrier()                                    # keeps V from being merged into the preparation
    if V is not None:
        qc.append(UnitaryGate(V, label="V_QI"), [qr[0]])
    rotate_to_z(qc, qr[0], basis)
    qc.measure(qr[0], cr[0])
    return qc

circuits, index = [], []
for d in DELAYS_DT:
    for s in VAL_INPUTS:
        for b in "XYZ":
            circuits.append(delay_circuit(s, b, d)); index.append((d, s, b))
counts_raw = run_batch(circuits, "characterisation+raw", physical_qubits=[QUBIT])
RAW = {}
for (d, s, b), cnt in zip(index, counts_raw):
    RAW.setdefault(d, {}).setdefault(s, {})[b] = cnt

CHANNEL = {}
for d in DELAYS_DT:
    r = {s: bloch_from_counts([RAW[d][s][b] for b in "XYZ"]) for s in VAL_INPUTS}
    M, c = affine_from_tomography(r["0"][0], r["1"][0], r["+"][0], r["+y"][0])
    M6, c6 = affine_from_pairs([PAULI_STATES[s][1] for s in VAL_INPUTS], [r[s][0] for s in VAL_INPUTS])
    qi = quasi_inverse(M, c)
    CHANNEL[d] = dict(M=M, c=c, M_lsq6=M6, c_lsq6=c6, bloch=r, qi=qi)
    print(f"\n=== delay {d} dt ===")
    print("M =\n", np.round(M, 4), "\nc =", np.round(c, 4), "   (6-state least squares: Tr M =", round(np.trace(M6), 4), ")")
    print(f"D2_raw = {qi['D2_raw']:.4f}   lambda_max = {qi['lambda_max']:.4f}   predicted D2_QI = {qi['D2_corrected']:.4f}   "
          f"predicted reduction = {qi['delta_D2']:.4f}")
    print("x_opt =", np.round(qi["x"], 4))
    # sanity: a correction can never remove more than the raw MSTD
    assert qi["delta_D2"] <= qi["D2_raw"] + 1e-12


## Job 2 - validate the correction

In [ ]:
# ---- Job 2: corrected validation (same six inputs, V_QI inserted after the delay) ----
circuits, index = [], []
for d in DELAYS_DT:
    for s in VAL_INPUTS:
        for b in "XYZ":
            circuits.append(delay_circuit(s, b, d, V=CHANNEL[d]["qi"]["V"])); index.append((d, s, b))
counts_qi = run_batch(circuits, "corrected validation", physical_qubits=[QUBIT])
QI = {}
for (d, s, b), cnt in zip(index, counts_qi):
    QI.setdefault(d, {}).setdefault(s, {})[b] = cnt

import pandas as pd
rows = []
for d in DELAYS_DT:
    rq = {s: bloch_from_counts([QI[d][s][b] for b in "XYZ"]) for s in VAL_INPUTS}
    Mq, cq = affine_from_pairs([PAULI_STATES[s][1] for s in VAL_INPUTS], [rq[s][0] for s in VAL_INPUTS])
    CHANNEL[d]["bloch_qi"] = rq; CHANNEL[d]["M_qi_measured"] = Mq; CHANNEL[d]["c_qi_measured"] = cq
    CHANNEL[d]["D2_qi_measured"] = mstd(Mq, cq)
    for s in VAL_INPUTS:
        r_in = PAULI_STATES[s][1]
        rr, se_r = CHANNEL[d]["bloch"][s]; rc, se_c = rq[s]
        rows.append(dict(delay_dt=d, input=s,
                         D_raw=trace_distance(r_in, rr), D_raw_se=trace_distance_se(r_in, rr, se_r),
                         D_QI=trace_distance(r_in, rc),  D_QI_se=trace_distance_se(r_in, rc, se_c)))
TABLE = pd.DataFrame(rows)
pd.set_option("display.precision", 3)
display(TABLE)
print("\nSummary (Bloch-ball MSTD):")
for d in DELAYS_DT:
    q = CHANNEL[d]["qi"]
    print(f"delay {d:>6} dt: D2_raw = {q['D2_raw']:.4f}  predicted D2_QI = {q['D2_corrected']:.4f}  "
          f"MEASURED D2_QI = {CHANNEL[d]['D2_qi_measured']:.4f}   | mean D over 6 inputs: "
          f"{TABLE[TABLE.delay_dt==d].D_raw.mean():.3f} -> {TABLE[TABLE.delay_dt==d].D_QI.mean():.3f}")


## Figure

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(DELAYS_DT), figsize=(4.2 * len(DELAYS_DT), 3.6), sharey=True)
axes = np.atleast_1d(axes)
for ax, d in zip(axes, DELAYS_DT):
    t = TABLE[TABLE.delay_dt == d]; x = np.arange(len(t)); w = 0.38
    ax.bar(x - w/2, t.D_raw, w, yerr=t.D_raw_se, label="raw", capsize=3)
    ax.bar(x + w/2, t.D_QI,  w, yerr=t.D_QI_se,  label="with $V_{QI}$", capsize=3)
    ax.set_xticks(x); ax.set_xticklabels(t.input); ax.set_title(f"{d} dt"); ax.grid(alpha=.3, axis="y")
axes[0].set_ylabel("trace distance to input"); axes[0].legend()
plt.tight_layout(); plt.savefig("fig_single_delay_v2.pdf"); plt.show()


## Export results (send this file back)

In [ ]:
meta = backend_metadata(cal_backend, [QUBIT]); meta["simulator"] = USE_SIMULATOR; meta["shots"] = SHOTS
results = {"experiment": "single_delay_v2", "metadata": meta, "jobs": JOB_LOG, "delays_dt": DELAYS_DT,
           "channels": {str(d): {k: v for k, v in CHANNEL[d].items() if k not in ("bloch", "bloch_qi", "qi")}
                        | {"qi": {k: v for k, v in CHANNEL[d]["qi"].items()}}
                        | {"bloch_raw": {s: {"r": CHANNEL[d]["bloch"][s][0], "se": CHANNEL[d]["bloch"][s][1]} for s in VAL_INPUTS},
                           "bloch_qi":  {s: {"r": CHANNEL[d]["bloch_qi"][s][0], "se": CHANNEL[d]["bloch_qi"][s][1]} for s in VAL_INPUTS}}
                        for d in DELAYS_DT},
           "table": TABLE.to_dict(orient="records"),
           "raw_counts": {str(d): RAW[d] for d in DELAYS_DT}, "corrected_counts": {str(d): QI[d] for d in DELAYS_DT}}
save_results(results, f"results_single_delay_{meta['backend'].replace('(', '_').replace(')', '')}.json")
